<a href="https://colab.research.google.com/github/arpitiitb/docs-rag/blob/main/04_build_testset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 4: Build the test set (about 60-90 minutes)

**Where we are:** we have a working RAG system (Steps 1-3). But "it looks fine on a few questions" is not evidence. Step 4 builds the **test set** (also called a golden dataset) that every later step is measured against.

**Why this step matters most for your story.** Steps 5-9 compare dense search, hybrid search and a reranker. Without a fixed test set, you cannot say whether a change helped. With one, you can say "recall@5 went from X to Y", which is the kind of evidence that sets your project apart from most RAG demos.

**Each test item has:**

| Field | Meaning |
|---|---|
| `question` | what a user would ask |
| `reference_answer` | a short correct answer, used to grade the generated answer |
| `gold_url` | the page that contains the answer, used to grade retrieval |
| `evidence` | a short exact quote from that page, used to verify the item |
| `type` | `generated`, `paraphrase`, `exact_term` or `out_of_scope` |

**The plan:** the LLM drafts about 30 questions from sample pages, code checks them automatically, **you** review and fix them, then you add 8-10 questions of your own. Your review is the valuable part: a test set is only as good as its labels.

**Cost:** a few cents.

## 4.1 Setup

In [1]:
%pip install -q -U openai pydantic
import os, sys, json, re, random
from pydantic import BaseModel
from openai import OpenAI

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/docs-rag"
else:
    BASE = "/content/docs-rag"
DATA = f"{BASE}/data"

if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

docs = json.load(open(f"{DATA}/docs.json"))
client = OpenAI()
MODEL = "gpt-4o-mini"
print(len(docs), "pages loaded")

Mounted at /content/drive
28 pages loaded


## 4.2 Choose pages to ask about

**Concepts**
- **Stratified sampling:** take pages from every section, so the test set covers the whole corpus instead of whichever section is biggest.
- Skip very short pages (under 150 words). They are usually link lists with nothing worth asking.
- A fixed **random seed** makes the sample reproducible: run it again and you get the same pages.

In [2]:
random.seed(42)
PAGES_PER_SECTION = 5
by_section = {}
for url, d in docs.items():
    if d["words"] >= 150:
        by_section.setdefault(d["section"], []).append(url)

sample = []
for section, urls in sorted(by_section.items()):
    picked = random.sample(urls, min(PAGES_PER_SECTION, len(urls)))
    sample += picked
    print(f"{section}: {len(picked)} of {len(urls)} eligible pages")
print("Total pages sampled:", len(sample))

ai-gateway: 5 of 15 eligible pages
catalog-explorer: 1 of 1 eligible pages
unity-catalog: 5 of 12 eligible pages
Total pages sampled: 11


## 4.3 Draft questions with the LLM

**Concepts**
- You are using the **strict structured output** as used in basics: a Pydantic schema guarantees each draft has a question, an answer and evidence.
- The prompt asks for **self-contained, specific questions** ("this page" makes no sense to a real user) and for one question per page that uses **different words** from the page. That second rule matters: LLM-written questions tend to copy the document's wording, which makes retrieval look better than it really is.
- **Evidence:** the model must copy a short exact quote that supports its answer. In 4.4, code checks the quote really appears on the page. That catches invented answers automatically.

In [3]:
class QA(BaseModel):
    question: str
    reference_answer: str
    evidence: str          # exact quote from the page

class QAList(BaseModel):
    items: list[QA]

PROMPT = '''You are writing test questions for a documentation search assistant.
Write exactly 2 questions that a Databricks user might ask, each answerable from the page below.
Rules:
- Each question must be self-contained and specific. Never say "this page", "the document" or "the article".
- Question 1 may use the page's wording. Question 2 must use different, everyday words than the page.
- reference_answer: 1-2 sentences, correct according to the page.
- evidence: copy an exact sentence or phrase (5-25 words) from the page that supports the answer.

Page title: {title}
Page text:
{text}'''

parse = getattr(client.chat.completions, "parse", None) or client.beta.chat.completions.parse

drafts = []
for url in sample:
    d = docs[url]
    c = parse(model=MODEL, temperature=0, response_format=QAList,
              messages=[{"role": "user", "content": PROMPT.format(title=d["title"], text=d["text"][:6000])}])
    msg = c.choices[0].message
    if msg.refusal or msg.parsed is None:
        print("skipped:", d["title"])
        continue
    for n, item in enumerate(msg.parsed.items[:2]):
        drafts.append({"question": item.question, "reference_answer": item.reference_answer,
                       "evidence": item.evidence, "gold_url": url, "title": d["title"],
                       "type": "generated" if n == 0 else "paraphrase"})
print(len(drafts), "draft questions")
for x in drafts[:4]:
    print(f"\n[{x['type']}] {x['question']}\n  answer: {x['reference_answer']}")

22 draft questions

[generated] How can you configure traffic splitting for models in Databricks?
  answer: Traffic splitting can be configured in the UI by selecting the model service and adding destination entries for each model backend, setting the traffic percentage for each.

[paraphrase] What happens if a request fails when using fallbacks in Databricks?
  answer: If a request fails, the system retries it against the configured fallback destinations in the order you defined until one succeeds or all options are exhausted.

[generated] What command do you use to launch Codex CLI in Databricks?
  answer: To launch Codex CLI, use the command 'ug codex'.

[paraphrase] How can I pick an open-source model when using GitHub Copilot?
  answer: You can select an open-source model from the Databricks group in the Copilot Chat model picker after launching GitHub Copilot CLI.


## 4.4 Automatic checks

**Concepts:** let code catch the obvious problems so your review time goes to judgment calls.
- **Evidence check:** is the quote actually on the gold page? (Compared after lowercasing and collapsing spaces.) If not, the model may have invented or paraphrased it; review those carefully.
- **Banned phrases:** questions mentioning "this page" or "the document".
- **Near-duplicate questions:** two questions with mostly the same words.

In [4]:
def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

BANNED = ["this page", "the page", "the document", "this document", "the article", "this article"]

def words(s):
    return set(re.findall(r"\w+", s.lower()))

for i, x in enumerate(drafts):
    x["evidence_found"] = norm(x["evidence"]) in norm(docs[x["gold_url"]]["text"])
    x["banned_phrase"] = any(b in x["question"].lower() for b in BANNED)
    x["near_duplicate_of"] = None
    for j in range(i):
        a, b = words(x["question"]), words(drafts[j]["question"])
        if a and len(a & b) / len(a | b) > 0.7:
            x["near_duplicate_of"] = j
            break

print("evidence found on page:", sum(x["evidence_found"] for x in drafts), "of", len(drafts))
print("banned phrases:", sum(x["banned_phrase"] for x in drafts))
print("near duplicates:", sum(x["near_duplicate_of"] is not None for x in drafts))
json.dump(drafts, open(f"{DATA}/testset_draft.json", "w"), indent=2)

evidence found on page: 21 of 22
banned phrases: 0
near duplicates: 0


**What you should see:** most evidence quotes found (often 70-95%) and few or no banned phrases or duplicates. The draft is saved, so you won't lose it if Colab disconnects.

## 4.5 Review each question yourself

**How to judge an item (about 30 seconds each):**
- **Keep** if the question is clear, a real user might ask it, only one page answers it, and the answer is correct.
- **Drop** if it is vague, trivial, answerable from many pages, or wrong.
- **Edit** if it is nearly right: fix the wording or the answer.

Items marked `⚠` failed an automatic check: look harder at those. The cell asks for your choice one item at a time: type `k` (keep), `d` (drop) or `e` (edit), then press Enter. For an edit, press Enter on a field to leave it unchanged.

Your choices are saved after every item, so you can stop and rerun the cell later; it continues where you left off.

In [5]:
REVIEW_PATH = f"{DATA}/testset_reviewed.json"
reviewed = json.load(open(REVIEW_PATH)) if os.path.exists(REVIEW_PATH) else {}

for i, x in enumerate(drafts):
    if str(i) in reviewed:
        continue
    flags = []
    if not x["evidence_found"]: flags.append("evidence NOT found on page")
    if x["banned_phrase"]: flags.append("mentions 'this page'")
    if x["near_duplicate_of"] is not None: flags.append(f"similar to #{x['near_duplicate_of']}")
    print("=" * 90)
    print(f"#{i} [{x['type']}]  page: {x['title']}")
    if flags: print("⚠", "; ".join(flags))
    print("Q:", x["question"])
    print("A:", x["reference_answer"])
    print("Evidence:", x["evidence"])
    choice = input("k = keep, d = drop, e = edit: ").strip().lower()
    item = dict(x)
    if choice == "d":
        item = None
    elif choice == "e":
        q = input("New question (Enter = keep): ").strip()
        a = input("New answer (Enter = keep): ").strip()
        if q: item["question"] = q
        if a: item["reference_answer"] = a
    reviewed[str(i)] = item
    json.dump(reviewed, open(REVIEW_PATH, "w"), indent=2)

kept = [v for v in reviewed.values() if v]
print(f"\nReview done: kept {len(kept)} of {len(reviewed)}")

#0 [generated]  page: Configure routing and fallbacks for models | Databricks on AWS
Q: How can you configure traffic splitting for models in Databricks?
A: Traffic splitting can be configured in the UI by selecting the model service and adding destination entries for each model backend, setting the traffic percentage for each.
Evidence: In your Databricks workspace, click AI Gateway in the sidebar and select the model service you want to edit.
k = keep, d = drop, e = edit: k
#1 [paraphrase]  page: Configure routing and fallbacks for models | Databricks on AWS
Q: What happens if a request fails when using fallbacks in Databricks?
A: If a request fails, the system retries it against the configured fallback destinations in the order you defined until one succeeds or all options are exhausted.
Evidence: The system attempts fallbacks sequentially until one succeeds or it exhausts all fallback options.
k = keep, d = drop, e = edit: k
#2 [generated]  page: Use open-source models | Databricks

## 4.6 Add your own questions

**Concepts:** LLM drafts are easy for retrieval because they copy the docs' wording. Your own questions test what real users do.
- **Paraphrase** (3-4): ask the way a colleague would, with no words copied from the page. Example idea: "Who can see my tables by default?"
- **Exact term** (2-3): use a specific name, code or setting from the docs. Keyword search usually beats embeddings on these, which you will show in Step 6.
- **Out of scope** (2-3): questions the docs don't answer, with `gold_url = None`. The system should say "I don't know". Keep them plausible, such as a question about a different cloud product.

Use `find_pages` to look up the right `gold_url`. It searches page titles and text for a keyword.

In [6]:
def find_pages(keyword, n=8):
    kw = keyword.lower()
    hits = [(d["text"].lower().count(kw), d["title"], u) for u, d in docs.items() if kw in d["text"].lower()]
    for count, title, url in sorted(hits, reverse=True)[:n]:
        print(f"{count:>3} mentions  {title}\n             {url}")

find_pages("external location")

 39 mentions  Unity Catalog best practices | Databricks on AWS
             https://docs.databricks.com/aws/en/data-governance/unity-catalog/best-practices
 13 mentions  Unity Catalog securable objects reference | Databricks on AWS
             https://docs.databricks.com/aws/en/data-governance/unity-catalog/securable-objects
  8 mentions  Upgrade a Databricks workspace to Unity Catalog | Databricks on AWS
             https://docs.databricks.com/aws/en/data-governance/unity-catalog/upgrade/
  3 mentions  Access control in Unity Catalog | Databricks on AWS
             https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/
  1 mentions  What is Unity Catalog? | Databricks on AWS
             https://docs.databricks.com/aws/en/data-governance/unity-catalog/
  1 mentions  What is Catalog Explorer? | Databricks on AWS
             https://docs.databricks.com/aws/en/catalog-explorer/
  1 mentions  Unity Catalog setup guide | Databricks on AWS
             https://

In [8]:
my_questions = [
    # --- Paraphrase (No words copied directly from the page) ---
    {
        "question": "Where should I go to define a path pointing to an S3 bucket in Unity Catalog?",
        "reference_answer": "You should create an external location object to reference an S3 bucket path inside Unity Catalog.",
        "gold_url": "https://docs.databricks.com/aws/en/data-governance/unity-catalog/best-practices",
        "type": "paraphrase"
    },
    {
        "question": "Is there a central place to see my workspace assets and structure in the interface?",
        "reference_answer": "Yes, Catalog Explorer provides a user interface to explore and manage your data schemas, tables, and permissions.",
        "gold_url": "https://docs.databricks.com/aws/en/catalog-explorer/",
        "type": "paraphrase"
    },
    {
        "question": "How do I authorize a specific user group to run queries using AI gateway endpoints?",
        "reference_answer": "You can control access by applying grant policies to model services within the AI Gateway.",
        "gold_url": "https://docs.databricks.com/aws/en/ai-gateway/govern-access-to-models-with-grant-policies",
        "type": "paraphrase"
    },
    # --- Exact Term (Using highly specific terms, settings or commands) ---
    {
        "question": "Which securable object is the parent of catalogs in the Unity Catalog object hierarchy?",
        "reference_answer": "The metastore is the top-level container for metadata and the parent of catalogs in Unity Catalog.",
        "gold_url": "https://docs.databricks.com/aws/en/data-governance/unity-catalog/securable-objects",
        "type": "exact_term"
    },
    {
        "question": "What is the main difference between managed and external tables in Unity Catalog when dropping them?",
        "reference_answer": "Dropping a managed table deletes both its metadata and the underlying files from your cloud storage, while dropping an external table only removes the metadata, leaving the actual files intact.",
        "gold_url": "https://docs.databricks.com/aws/en/data-governance/unity-catalog/managed-versus-external",
        "type": "exact_term"
    },
    {
        "question": "What is the name of the tool or CLI command used to upgrade schemas to Unity Catalog?",
        "reference_answer": "You can use the Upgrade Wizard in Catalog Explorer to upgrade your non-Unity Catalog tables.",
        "gold_url": "https://docs.databricks.com/aws/en/data-governance/unity-catalog/upgrade/",
        "type": "exact_term"
    },
    # --- Out of Scope (Questions the documentation does not answer) ---
    {
        "question": "How do I configure AWS IAM Identity Center to sync groups with Google Cloud Identity?",
        "reference_answer": "I don't know based on the provided documentation.",
        "gold_url": None,
        "type": "out_of_scope"
    },
    {
        "question": "Can I use Azure Data Factory to directly load tables into Databricks without Unity Catalog?",
        "reference_answer": "I don't know based on the provided documentation.",
        "gold_url": None,
        "type": "out_of_scope"
    }
]

for q in my_questions:
    assert q["gold_url"] is None or q["gold_url"] in docs, f"gold_url not in corpus: {q['gold_url']}"
    q.setdefault("evidence", "")
    q["title"] = docs[q["gold_url"]]["title"] if q["gold_url"] else None
print(len(my_questions), "of your own questions added")

8 of your own questions added


## 4.7 Combine, check and save the final test set

**Concepts:** the test set is now **frozen**. Do not edit it to make results look better later. If you must change it, save a new version and note why. Every result table must say which version it used.

In [9]:
from collections import Counter
FIELDS = ["question", "reference_answer", "gold_url", "title", "evidence", "type"]
testset = [{k: q.get(k) for k in FIELDS} for q in kept + my_questions]
for i, q in enumerate(testset):
    q["id"] = i

print("Total test questions:", len(testset))
print("By type:", dict(Counter(q["type"] for q in testset)))
sections = Counter(docs[q["gold_url"]]["section"] for q in testset if q["gold_url"])
print("By section:", dict(sections))

json.dump({"version": 1, "n": len(testset), "items": testset}, open(f"{DATA}/testset.json", "w"), indent=2)
print("Saved", f"{DATA}/testset.json")

Total test questions: 21
By type: {'generated': 8, 'paraphrase': 8, 'exact_term': 3, 'out_of_scope': 2}
By section: {'ai-gateway': 8, 'catalog-explorer': 2, 'unity-catalog': 9}
Saved /content/drive/MyDrive/docs-rag/data/testset.json
